# **Student_Performance_Prediction**

1.Importing Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# ML libraries
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier


2.Load the Dataset

In [ ]:
df = pd.read_csv("/content/StudentsPerformance.csv")
df.head()

3.Basic Checks

In [ ]:
print("Shape of the DataFrame:")
df.shape

In [ ]:
print("\nInformation about the DataFrame (data types, non-null values):")
df.info()

In [ ]:
print("\nDescriptive statistics of the DataFrame:")
df.describe()

In [ ]:
print("\nNumber of null values per column:")
df.isnull().sum()

**4.Exploratory Data Analysis**

In [ ]:
df['math score'].hist()
plt.title("Math Score Distribution")
plt.show()

In [ ]:
df['reading score'].hist()
plt.title("Reading Score Distribution")
plt.show()

In [ ]:
df['writing score'].hist()
plt.title("Writing Score Distribution")
plt.show()

In [ ]:
sns.boxplot(x='gender', y='math score', data=df)
plt.title("Gender vs Math Score")
plt.show()

In [ ]:
sns.boxplot(x='test preparation course', y='math score', data=df)
plt.title("Test Preparation vs Math Score")
plt.show()

In [ ]:
sns.heatmap(df[['math score','reading score','writing score']].corr(), annot=True)
plt.title("Correlation Heatmap")
plt.show()

In [ ]:
sns.boxplot(x='lunch', y='math score', data=df)
plt.title("Lunch vs Performance")
plt.show()

**Insights:
    Most students score in the mid-range (60–80).
    Test preparation has a strong positive impact.
    Reading and writing scores are highly correlated.
    Socio-economic factors influence performance.
    Female students perform better in language subjects.**

**5.Data Preprocessing**

In [ ]:
df['performance'] = df['math score'].apply(lambda x: 1 if x >= 50 else 0)
df.head()

In [ ]:
df = pd.get_dummies(df, drop_first=True)
df.head()

In [ ]:
df.drop(['math score'], axis=1, inplace=True)

In [ ]:
df.head()

In [ ]:
df.shape

**6.Train_Test Split**

In [ ]:
X = df.drop('performance', axis=1)
y = df['performance']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

**7.Model Training**

In [ ]:
models = {
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(random_state=42)
}

In [ ]:
results = []

for name, model in models.items():
    model.fit(X_train, y_train)

    train_preds = model.predict(X_train)
    preds = model.predict(X_test)

    train_acc = accuracy_score(y_train, train_preds)
    acc = accuracy_score(y_test, preds)
    f1 = f1_score(y_test, preds)

    results.append((name, train_acc, acc, f1))

    print(f"{name}")
    print(f"Train Accuracy: {train_acc:.4f}")
    print(f"Test Accuracy:  {acc:.4f}")
    print(f"F1 Score:       {f1:.4f}")
    print("-" * 40)

results_df = pd.DataFrame(results, columns=["Model", "Train Accuracy", "Test Accuracy", "F1 Score"])
results_df

**8.Feature Importance**

In [ ]:
model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)

importances = model.feature_importances_

# Convert to dataframe
feat_imp = pd.Series(importances, index=X.columns)
feat_imp = feat_imp.sort_values(ascending=True)

# Plot
feat_imp.plot(kind='barh')
plt.title("Feature Importance")
plt.show()

**9.Confusion Matrix**

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns

best_model = LogisticRegression(max_iter=1000)
best_model.fit(X_train, y_train)
preds = best_model.predict(X_test)

cm = confusion_matrix(y_test, preds)

sns.heatmap(cm, annot=True, fmt='d')
plt.title("Confusion Matrix")
plt.show()

**10.Cross Validation**

In [ ]:
from sklearn.model_selection import cross_val_score

model = LogisticRegression(max_iter=1000)

scores = cross_val_score(model, X, y, cv=5, scoring='f1')

print("F1 Scores:", scores)
print("Mean F1 Score:", scores.mean())

**11.Handle Class Imbalance**

In [ ]:
model_balanced = LogisticRegression(max_iter=1000, class_weight='balanced')

model_balanced.fit(X_train, y_train)
preds_balanced = model_balanced.predict(X_test)

from sklearn.metrics import classification_report
print(classification_report(y_test, preds_balanced))

**12.Pipeline**

In [ ]:
pipeline = Pipeline([
    ('model', LogisticRegression(max_iter=1000))
])

pipeline.fit(X_train, y_train)
preds_pipe = pipeline.predict(X_test)

print(f"Pipeline Accuracy: {accuracy_score(y_test, preds_pipe):.4f}")
print(f"Pipeline F1 Score: {f1_score(y_test, preds_pipe):.4f}")

**13.Hyperparameter Tunning**

In [ ]:
from sklearn.model_selection import GridSearchCV

params = {
    'C': [0.01, 0.1, 1, 10],
    'solver': ['lbfgs', 'liblinear']
}
grid = GridSearchCV(LogisticRegression(max_iter=1000),
                    param_grid=params,
                    cv=5,
                    scoring='f1')

grid.fit(X_train, y_train)

print("Best Parameters:", grid.best_params_)
print("Best Score:", grid.best_score_)

In [ ]:
best_model = grid.best_estimator_

preds = best_model.predict(X_test)

print("Final Accuracy:", accuracy_score(y_test, preds))
print("Final F1:", f1_score(y_test, preds))

In [ ]:
print("Final Model Selected: Logistic Regression (Tuned)")
print("Reason: Best balance of accuracy and F1-score with stable performance")